# AcRanker 复现 (Eitzinger et al., NAR 2020)

**方法**：XGBoost 二分类器 + 纯序列特征

**特征**：
- 氨基酸组成 (AAC, 20d)
- 基于7组理化性质分组的 2-mer 频率 (49d)
- 基于7组理化性质分组的 3-mer 频率 (343d)
- 总计 412 维特征

**参考**：https://github.com/amina01/AcRanker

**环境**：`lm-hf`（需额外安装 `xgboost`）

In [1]:
import os, json
import numpy as np
import pandas as pd
from itertools import product
from sklearn.preprocessing import normalize
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score, matthews_corrcoef, roc_auc_score,
)
from xgboost import XGBClassifier

BENCHMARKS_DIR = '/home/nemophila/projects/protein_bert/anticrispr_benchmarks'
RESULTS_DIR   = '/home/nemophila/projects/protein_bert/Comparison/results'
SEED = 22

## 1. 加载数据（与融合模型相同的 train/test 划分）

In [2]:
train_df = pd.read_csv(f'{BENCHMARKS_DIR}/anticrispr_binary.train.csv').dropna().drop_duplicates().reset_index(drop=True)
test_df  = pd.read_csv(f'{BENCHMARKS_DIR}/anticrispr_binary.test.csv').dropna().drop_duplicates().reset_index(drop=True)
print('train:', train_df.shape, 'test:', test_df.shape)
print('label distribution (train):', train_df['label'].value_counts().to_dict())
print('label distribution (test):', test_df['label'].value_counts().to_dict())

train: (1107, 2) test: (286, 2)
label distribution (train): {0: 902, 1: 205}
label distribution (test): {0: 260, 1: 26}


## 2. AcRanker 特征提取

与原始 AcRanker 代码一致：将 20 种氨基酸按理化性质映射为 7 组，计算 AAC + grouped 2-mer + grouped 3-mer。

In [3]:
AA_LIST = list('ACDEFGHIKLMNPQRSTVWY')

# AcRanker 的 7 组理化性质分组（与原始 server2.py 完全一致）
GROUPS = {
    'A': '1', 'V': '1', 'G': '1',
    'I': '2', 'L': '2', 'F': '2', 'P': '2',
    'Y': '3', 'M': '3', 'T': '3', 'S': '3',
    'H': '4', 'N': '4', 'Q': '4', 'W': '4',
    'R': '5', 'K': '5',
    'D': '6', 'E': '6',
    'C': '7',
}


def amino_acid_composition(seq: str) -> np.ndarray:
    """20-d 氨基酸组成比例（与 server2.py 的 get_amino_acids_percent 对齐）。"""
    counts = {aa: 0 for aa in AA_LIST}
    for ch in seq:
        if ch in counts:
            counts[ch] += 1
    total = max(len(seq), 1)
    return np.array([counts[aa] / total for aa in AA_LIST], dtype=np.float64)


def grouped_kmer(seq: str, k: int) -> np.ndarray:
    """基于 7 组理化性质分组的 k-mer 频率向量。
    注意：原始代码对 2-mer 和 3-mer 都除以 len(s)-1（不是 len(s)-k+1），严格保持一致。
    """
    digits = '1234567'
    combos = [''.join(p) for p in product(digits, repeat=k)]
    combo_idx = {int(c): i for i, c in enumerate(combos)}
    vec = np.zeros(len(combos), dtype=np.float64)

    # 处理未知氨基酸：找到序列中出现最多的组，用于替换（与原始 except 分支一致）
    group_counts = {d: 0 for d in digits}
    for ch in seq:
        g = GROUPS.get(ch)
        if g:
            group_counts[g] += 1
    fallback = max(group_counts, key=group_counts.get)

    for j in range(len(seq) - k + 1):
        kmer = seq[j:j + k]
        code = ''
        for ch in kmer:
            code += GROUPS.get(ch, fallback)
        vec[combo_idx[int(code)]] += 1

    # 原始代码: V=V/(len(s)-1)，对 2-mer 和 3-mer 都用 len(s)-1
    denom = max(len(seq) - 1, 1)
    return vec / denom


def extract_acranker_features(seq: str) -> np.ndarray:
    """提取 AcRanker 的 412 维特征（与 server2.py 的 prot_feats_seq 对齐）。"""
    aac = amino_acid_composition(seq)
    aac = normalize(aac.reshape(1, -1), norm='l2')[0]
    tmer = grouped_kmer(seq, 2)
    tmer = normalize(tmer.reshape(1, -1), norm='l2')[0]
    thmer = grouped_kmer(seq, 3)
    thmer = normalize(thmer.reshape(1, -1), norm='l2')[0]
    return np.concatenate([aac, tmer, thmer])


print('Feature dimension:', len(extract_acranker_features('ACDEFGHIKLMNPQRSTVWY')))

Feature dimension: 412


In [4]:
X_train = np.array([extract_acranker_features(s) for s in train_df['seq']])
X_test  = np.array([extract_acranker_features(s) for s in test_df['seq']])
y_train = train_df['label'].to_numpy(dtype=int)
y_test  = test_df['label'].to_numpy(dtype=int)
print('X_train:', X_train.shape, 'X_test:', X_test.shape)

X_train: (1107, 412) X_test: (286, 412)


## 3. 训练 XGBoost 二分类器

In [5]:
from sklearn.model_selection import RandomizedSearchCV

# 限制总并行度在 36 以下，避免把整机 CPU 打满
MAX_CPU_JOBS = 32

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.2, 0.3],
}

base_xgb = XGBClassifier(
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    random_state=SEED,
    eval_metric='logloss',
    n_jobs=1,  # 每个 fit 单线程，避免和外层并行叠加
)

search = RandomizedSearchCV(
    base_xgb, param_dist,
    n_iter=80, scoring='roc_auc',
    cv=5, random_state=SEED, n_jobs=MAX_CPU_JOBS, verbose=1,
)
search.fit(X_train, y_train)
model = search.best_estimator_

print(f'Best params: {search.best_params_}')
print(f'Best CV AUC: {search.best_score_:.4f}')
print(f'Parallel jobs limit: {MAX_CPU_JOBS}')
print('Training done.')

Fitting 5 folds for each of 80 candidates, totalling 400 fits
Best params: {'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 7, 'max_depth': 6, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 0.7}
Best CV AUC: 0.8400
Parallel jobs limit: 32
Training done.


## 4. 评估

In [6]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ids = np.digitize(y_prob, bins) - 1
    ece = 0.0
    n = len(y_true)
    for b in range(n_bins):
        m = ids == b
        if np.any(m):
            ece += (np.sum(m) / n) * abs(float(np.mean(y_true[m])) - float(np.mean(y_prob[m])))
    return float(ece)


def evaluate_binary_full(y_true, y_prob, threshold=0.5):
    y_cls = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_cls).ravel()
    return {
        'AUC':  float(roc_auc_score(y_true, y_prob)),
        'AUPRC': float(average_precision_score(y_true, y_prob)),
        'F1':   float(f1_score(y_true, y_cls)),
        'MCC':  float(matthews_corrcoef(y_true, y_cls)),
        'Brier': float(brier_score_loss(y_true, y_prob)),
        'ECE':  expected_calibration_error(y_true, y_prob),
        'ACC':  float(accuracy_score(y_true, y_cls)),
        'SN':   float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0,
        'SP':   float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0,
        'Threshold': float(threshold),
    }


y_prob = model.predict_proba(X_test)[:, 1]

# 用验证集（从训练集中划分 10%）找最优阈值
from sklearn.model_selection import train_test_split
_, X_val, _, y_val = train_test_split(X_train, y_train, test_size=0.1, stratify=y_train, random_state=SEED)
val_prob = model.predict_proba(X_val)[:, 1]
best_thr, best_f1 = 0.5, 0.0
for t in np.arange(0.1, 0.9, 0.01):
    f = f1_score(y_val, (val_prob >= t).astype(int))
    if f > best_f1:
        best_f1, best_thr = f, t
print(f'Best threshold (val F1={best_f1:.4f}): {best_thr:.2f}')

metrics = evaluate_binary_full(y_test, y_prob, threshold=best_thr)
for k, v in metrics.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')

Best threshold (val F1=1.0000): 0.26
AUC: 0.8374
AUPRC: 0.3904
F1: 0.3514
MCC: 0.2811
Brier: 0.0731
ECE: 0.0555
ACC: 0.8322
SN: 0.5000
SP: 0.8654
Threshold: 0.2600


In [7]:
# 保存结果
result = {'method': 'AcRanker (XGBoost)', 'metrics': metrics}
with open(f'{RESULTS_DIR}/acranker_metrics.json', 'w') as f:
    json.dump(result, f, indent=2)
np.savez(f'{RESULTS_DIR}/acranker_predictions.npz', y_true=y_test, y_prob=y_prob)
print('Results saved.')

Results saved.
